<td>
<a href="https://colab.research.google.com/github/raoulg/MADS-DAV/blob/main/notebooks/lesson5/05.3-notebook-to-script.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
</td>


# 5.3 From notebook to script — and what a residual is for

Every notebook so far has been the whole story: cells run once, top to bottom, and the
narrative *is* the code. That stops working the moment you want to run the same analysis
twice — on tomorrow's data, on a schedule, from a `Makefile` — because a cell has no
natural boundary. Nothing stops one cell's variable from leaking into the next, and
nothing forces the logic small enough to test or reuse.

This notebook is deliberately **not** about chat data. `scripts/covid_pipeline.py` runs
the same loop on public Dutch COVID figures — config, process, model, residual, distribution
fit — as an ordinary Python script with a `main()`. The domain is different on purpose:
if the loop only makes sense wrapped around WhatsApp messages, it was never really a loop,
it was a plot. Watch for how little of what follows is about COVID.


In [ ]:
import numpy as np
import pandas as pd
from goad_toolkit.models import linear_model, mse, train_model
from goad_toolkit.visualizer import ComparePlot, PlotSettings, ResidualPlot

from scripts.covid_pipeline import VACCINATION_START, fit_linear, plot_model, plot_residual, plot_zscores, preprocess
from wa_analyzer.data import load_showcase


## The loop, as five functions

`preprocess`, `plot_zscores`, `fit_linear`, `plot_model`, `plot_residual` — each one does
one thing, takes a DataFrame or returns one, and has a name a reader can guess the contents
of. That is the entire trick. None of them is more than eight lines; the trick is that
they are *separate* lines, in a file, not cells.


In [ ]:
data = preprocess()
fig, ax = plot_zscores(data)


Deaths, shifted back by the average reporting lag, against positive tests — both scaled
to the same axis so their shapes are comparable. They track each other closely, which is
exactly the assumption a linear model needs: a roughly constant ratio of deaths to cases.


In [ ]:
data = fit_linear(data)
fig, ax = plot_model(data)


In [ ]:
fig, ax = plot_residual(data)


## What the residual says

Nothing in that model knows a vaccine exists. It fits one ratio of deaths to positive
tests across the entire window, and the residual is where that assumption breaks: strongly
positive through the second wave — the model under-predicts, because deaths keep pace with
a ratio the raw case count alone can't explain (a lagging wave still climbing) — then,
right around the vaccination line, it flips and stays negative for months. After
vaccination, the same number of positive tests produces fewer deaths, and a model with one
fixed ratio has no way to say so.

**The residual is not leftover noise here — it is the finding.** It told you exactly where
the model's assumption stopped holding, and roughly when.


## Improving the model

The residual named the problem: one ratio, two regimes. The fix costs nothing new — the
same `linear_model` / `mse` / `train_model` loop, fit twice, split at the date the residual
pointed at.


In [ ]:
before = data.index < VACCINATION_START
after = ~before


def fit_segment(subset: pd.DataFrame) -> list[float]:
    x = subset["positivetests"].to_numpy()
    y = subset["deaths"].to_numpy()
    return train_model(x, y, linear_model, mse, [0.01, 1.0], bounds=[(0, 1.0), (0, None)])


params_before, params_after = fit_segment(data[before]), fit_segment(data[after])
print(f"before vaccination: {params_before}")
print(f"after vaccination:  {params_after}")

data["predicted deaths (2-segment)"] = np.where(
    before,
    linear_model(data["positivetests"].to_numpy(), params_before),
    linear_model(data["positivetests"].to_numpy(), params_after),
)
data["residual (2-segment)"] = data["deaths_shifted"] - data["predicted deaths (2-segment)"]

for label, mask in [("before vaccination", before), ("after vaccination", after)]:
    single = data.loc[mask, "residual"]
    split = data.loc[mask, "residual (2-segment)"]
    print(f"{label:>18s}  single: mean|error| {single.abs().mean():5.2f}   "
          f"2-segment: mean|error| {split.abs().mean():5.2f}")


**The after-vaccination slope is zero** — not a small number, the fit's actual answer.
Positive tests stopped predicting deaths through this ratio at all, and the optimiser
fell back to a flat average rather than force a slope that didn't fit.

Splitting the model **fixes the part of the problem that was fixable this way**: before
vaccination, the mean error drops by nearly half, because the single model's one ratio was
too low for the growing wave. **After vaccination it barely moves.** That is not a bug in
the split — it is the split correctly reporting that a second ratio does not fix this half
either. Deaths kept declining through the summer for reasons the case count had stopped
tracking at all; no straight line through `positivetests`, with any slope, explains that
(checked — even freeing the slope from its non-negative bound barely helps). Fixing that
half would need a different predictor, most plausibly time itself, not another ratio.

**Knowing which half improved and which didn't is worth more than the single combined
number.** It says exactly what to try next, instead of "try harder."


In [ ]:
settings = PlotSettings(figsize=(12, 6), title="Residual, single model vs. two segments",
                        xlabel="date", ylabel="error")
host = ResidualPlot(settings)
fig, axes = host.create_figure(n_plots=2)
host.plot_on_axes(ResidualPlot(settings), axes[0], data=data, x=data.index.name or "date",
                  y="residual", date=VACCINATION_START, datelabel="vaccination", interval=1)
axes[0].set_title("single model")
host.plot_on_axes(ResidualPlot(settings), axes[1], data=data, x=data.index.name or "date",
                  y="residual (2-segment)", date=VACCINATION_START, datelabel="vaccination", interval=1)
axes[1].set_title("two segments")
fig.tight_layout()


The left hump — the growing second wave, before vaccination — is visibly smaller in the
right panel: that is the bias the split actually fixed. The long negative drift after the
vaccination line is **still there, at close to the same depth**, in both panels. The split
did not fix that half, and the picture says so as plainly as the per-segment numbers did.
A summary statistic averaged over the whole window would have hidden exactly this —
one real improvement and one non-improvement, netting out to "a bit better overall."


## One more basis function

Splitting a model in two worked because the *shape* the residual pointed at was a step —
one ratio, then another. Not every shape is a step. Monthly airline passenger counts, 1949
through 1960 — a trend, plus a cycle that repeats every 12 months — is 5.2's basis-function
question again, in its plainest form: a straight line is one basis function, and the data
very obviously has a second one built in.


In [ ]:
flights = load_showcase("flights")
months = pd.to_datetime(flights["year"].astype(str) + "-" + flights["month"], format="%Y-%b")
t = np.arange(len(flights)).astype(float)
y = flights["passengers"].to_numpy().astype(float)

trend_only = train_model(t, y, linear_model, mse, [1.0, 100.0])
yhat_trend = linear_model(t, trend_only)
print(f"trend only: {trend_only}")
print(f"variance explained: {1 - np.var(y - yhat_trend) / np.var(y):.1%}")


A straight line already explains most of the variance — air travel grew steadily for a
decade, so the trend is real. What it cannot do is explain *why* every twelfth point sits
above the line and every sixth or so sits below it: nothing in `a*t + b` has a notion of
"month". Write that notion down as its own term instead of hoping a better trend line
will absorb it.


In [ ]:
def trend_plus_seasonal(t: np.ndarray, params: list[float]) -> np.ndarray:
    a, b, amplitude, phase = params
    return a * t + b + amplitude * np.sin(2 * np.pi * t / 12 + phase)


trend_and_cycle = train_model(t, y, trend_plus_seasonal, mse, [1.0, 100.0, 30.0, 0.0])
yhat_cycle = trend_plus_seasonal(t, trend_and_cycle)
print(f"trend + sine(period=12): {trend_and_cycle}")
print(f"variance explained: {1 - np.var(y - yhat_cycle) / np.var(y):.1%}")

settings = PlotSettings(figsize=(10, 4.5), title="Airline passengers: trend alone vs. trend + a 12-month cycle",
                        xlabel="", ylabel="passengers")
compare = pd.DataFrame({"month": months, "trend only": yhat_trend, "trend + cycle": yhat_cycle, "actual": y})
fig, ax = ComparePlot(settings).plot(data=compare, x="month", y1="trend only", y2="trend + cycle")
ax.scatter(compare["month"], compare["actual"], s=8, color="black", alpha=0.5, label="actual", zorder=3)
ax.legend()


One added term — amplitude and phase of a 12-month sine, four parameters total — lifts
explained variance from 85% to 93%. The remaining gap is visible and honest: the real
cycle *widens* as the trend grows (this is the "multiplicative seasonality" the showcase
list warns about), and a sine added on top of a line cannot do that; it stays the same
width throughout. Getting the rest would mean multiplying the cycle by the trend instead
of adding it — a different, still simple, basis function, and a natural next question
rather than a failure of this one.

**Same loop as the COVID model, same lesson twice:** look at what the current basis
functions cannot explain, write down *why not* in one sentence, and that sentence is
usually the next term to add.


## What to carry forward

1. **A script is a notebook with the leakage removed.** Small functions, each independently
   callable, are what let you re-run one step, import it elsewhere, or test it without
   re-running everything above it.
2. **Fit the residual, not just the value.** The improvement in both examples above came
   from asking what the *error* still looked like, not from staring harder at the fit.
3. **A basis function is a claim about the shape you expect.** A step said "one ratio,
   then another." A sine said "this repeats every twelve months." Both are testable, and
   both were confirmed by the residual shrinking once they were added — not assumed because
   they seemed reasonable.
